In [16]:
from pathlib import Path

LABEL_ROOT = Path("dataset_final/labels")
NC = 42  # total number of classes

print("🔧 NORMALIZING LABEL CLASS IDS TO 0-BASED\n")

files_fixed = 0
labels_fixed = 0

for lbl_path in LABEL_ROOT.rglob("*.txt"):
    lines = lbl_path.read_text().splitlines()
    class_ids = []

    for ln in lines:
        parts = ln.split()
        if len(parts) != 5:
            continue
        class_ids.append(int(parts[0]))

    if not class_ids:
        continue

    max_id = max(class_ids)

    # Case 1: already 0-based
    if max_id <= NC - 1:
        continue

    # Case 2: 1-based (contains NC)
    if max_id == NC:
        new_lines = []
        for ln in lines:
            parts = ln.split()
            if len(parts) != 5:
                continue
            parts[0] = str(int(parts[0]) - 1)
            new_lines.append(" ".join(parts))
            labels_fixed += 1

        lbl_path.write_text("\n".join(new_lines) + "\n")
        files_fixed += 1
        print(f"✔ Fixed: {lbl_path}")
        continue

    # Case 3: truly invalid
    raise ValueError(
        f"❌ Invalid class IDs in {lbl_path}: max_id={max_id}"
    )

print("\n✅ NORMALIZATION COMPLETE")
print(f"Files fixed : {files_fixed}")
print(f"Labels fixed: {labels_fixed}")


🔧 NORMALIZING LABEL CLASS IDS TO 0-BASED

✔ Fixed: dataset_final\labels\test\514a1e9daece75270dd402c76d63f544_jpg.rf.e2de662b91ffd13542a3e4470239a85a.txt
✔ Fixed: dataset_final\labels\test\bakwan_train-13-_jpg.rf.bc5f8fbeb72d2c03c26f4e762aa89443.txt
✔ Fixed: dataset_final\labels\test\bakwan_train-2-_jpg.rf.f37a2a1624b1e29380c5474b0a1d1fe5.txt
✔ Fixed: dataset_final\labels\test\bakwan_train-22-_jpg.rf.360d71d5e2738522dfa00b7eb5697ea4.txt
✔ Fixed: dataset_final\labels\test\BJ-173-_jpg.rf.09a4202b9125a2d3fc508bebddafed3c.txt
✔ Fixed: dataset_final\labels\test\BJ-200-_jpg.rf.0e75cfb033819eb963b57961003dab4c.txt
✔ Fixed: dataset_final\labels\test\BJ-234-_jpg.rf.0302a43a16df4fc6c2bb4245bc70e347.txt
✔ Fixed: dataset_final\labels\test\BJ-24-_jpg.rf.263046907aabb31e84cc32ededead877.txt
✔ Fixed: dataset_final\labels\test\BJ-278-_jpg.rf.7c307eacbfc330c6678236d62c9ac948.txt
✔ Fixed: dataset_final\labels\test\BJ-396-_jpg.rf.302f5cf910820b83399d6dd53e9b3348.txt
✔ Fixed: dataset_final\labels\test\BJ-

In [17]:
import os
import shutil

ROOT = r"C:\Users\nicho\Dokumen\Projects\FOOD\dataset_final"

splits = ["train", "val", "test"]

for s in splits:
    # Create target dirs
    os.makedirs(os.path.join(ROOT, s, "images"), exist_ok=True)
    os.makedirs(os.path.join(ROOT, s, "labels"), exist_ok=True)

    # Move images
    src_img = os.path.join(ROOT, "images", s)
    dst_img = os.path.join(ROOT, s, "images")
    if os.path.exists(src_img):
        for f in os.listdir(src_img):
            shutil.move(os.path.join(src_img, f), dst_img)

    # Move labels
    src_lbl = os.path.join(ROOT, "labels", s)
    dst_lbl = os.path.join(ROOT, s, "labels")
    if os.path.exists(src_lbl):
        for f in os.listdir(src_lbl):
            shutil.move(os.path.join(src_lbl, f), dst_lbl)

print("✅ Dataset structure fixed to YOLO standard")


✅ Dataset structure fixed to YOLO standard


In [18]:
import os

# Absolute path to dataset root
base_path = os.path.abspath("./dataset_final")

# Class names
classes = [f"class_{i}" for i in range(42)]

# =========================
# CREATE data.yaml
# =========================
yaml_content = f"""path: {base_path}
train: train/images
val: val/images
test: test/images

nc: 42
names: {classes}
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)

print("\n✓ data.yaml created successfully")

# =========================
# VERIFY PATHS
# =========================
print("\nVerifying paths...")
print("Train images:", os.path.exists(os.path.join(base_path, "train/images")))
print("Val images:  ", os.path.exists(os.path.join(base_path, "val/images")))
print("Test images:", os.path.exists(os.path.join(base_path, "test/images")))



✓ data.yaml created successfully

Verifying paths...
Train images: True
Val images:   True
Test images: True


In [19]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  

results = model.train(
    data="data.yaml",
    epochs=100,
    patience=15,            
    
    imgsz=640,
    batch=16,                
    device=0,
    workers=12,
    cache='disk',
    
    optimizer="AdamW",
    lr0=0.001,                 # Explicit learning rate
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=10,
    
    cos_lr=True,
    
    seed=42,
    verbose=True,
    deterministic=True
)

Ultralytics 8.3.237  Python-3.10.9 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train9, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=15, perspective=0.0, plots=True, p

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000001A6BD46B010>
Traceback (most recent call last):
  File "c:\Code\Python 3.10.9\environments\compvis\lib\site-packages\torch\utils\data\dataloader.py", line 1477, in __del__
    self._shutdown_workers()
  File "c:\Code\Python 3.10.9\environments\compvis\lib\site-packages\torch\utils\data\dataloader.py", line 1435, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 4.7it/s 7.1s0.2s
                   all       1050       1697      0.254      0.178      0.127      0.077

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100         4G      1.235      2.321      1.621         53        640: 100% ━━━━━━━━━━━━ 307/307 4.6it/s 1:06<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 5.0it/s 6.5s0.2s
                   all       1050       1697      0.447      0.268      0.239       0.14

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100         4G      1.224      2.237      1.602         53        640: 100% ━━━━━━━━━━━━ 307/307 4.6it/s 1:07<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 4.9it/s 6.7s0.2s
                   all 

In [24]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ============================================
# FIND YOUR TRAINING RUN
# ============================================
print("="*60)
print("CHECKING TRAINING RESULTS")
print("="*60)

# Find the most recent training run
train_dir = "./runs/detect/train9/"
runs = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

if not runs:
    print("❌ No training runs found!")
    exit(1)


run_path = "./runs/detect/train9/"

print(f"Location: {run_path}\n")

# ============================================
# READ TRAINING RESULTS
# ============================================
results_csv = f"{run_path}/results.csv"

if not os.path.exists(results_csv):
    print("❌ results.csv not found!")
    exit(1)

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()  # Remove whitespace

print("="*60)
print("TRAINING SUMMARY")
print("="*60)

# Get final metrics
final_row = df.iloc[-1]

print(f"\nTraining Details:")
print(f"  Total epochs completed: {len(df)}")
print(f"  Final epoch: {int(final_row['epoch'])}")

print(f"\nFinal Metrics:")
print(f"  mAP@0.5: {final_row['metrics/mAP50(B)']:.4f} ({final_row['metrics/mAP50(B)']*100:.2f}%)")
print(f"  mAP@0.5:0.95: {final_row['metrics/mAP50-95(B)']:.4f} ({final_row['metrics/mAP50-95(B)']*100:.2f}%)")
print(f"  Precision: {final_row['metrics/precision(B)']:.4f}")
print(f"  Recall: {final_row['metrics/recall(B)']:.4f}")

print(f"\nFinal Losses:")
print(f"  Box loss: {final_row['train/box_loss']:.4f}")
print(f"  Class loss: {final_row['train/cls_loss']:.4f}")
print(f"  DFL loss: {final_row['train/dfl_loss']:.4f}")

# Find best mAP
best_idx = df['metrics/mAP50(B)'].idxmax()
best_row = df.iloc[best_idx]

print(f"\nBest Performance:")
print(f"  Best mAP@0.5: {best_row['metrics/mAP50(B)']:.4f} at epoch {int(best_row['epoch'])}")
print(f"  Best mAP@0.5:0.95: {best_row['metrics/mAP50-95(B)']:.4f}")

# ============================================
# PLOT TRAINING CURVES
# ============================================
print("\n" + "="*60)
print("GENERATING PLOTS")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: mAP curves
axes[0, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', linewidth=2)
axes[0, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('mAP')
axes[0, 0].set_title('Validation mAP over Training')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Precision & Recall
axes[0, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', linewidth=2)
axes[0, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title('Precision & Recall')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Training losses
axes[1, 0].plot(df['epoch'], df['train/box_loss'], label='Box Loss', linewidth=2)
axes[1, 0].plot(df['epoch'], df['train/cls_loss'], label='Class Loss', linewidth=2)
axes[1, 0].plot(df['epoch'], df['train/dfl_loss'], label='DFL Loss', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Training Losses')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Validation losses
axes[1, 1].plot(df['epoch'], df['val/box_loss'], label='Box Loss', linewidth=2)
axes[1, 1].plot(df['epoch'], df['val/cls_loss'], label='Class Loss', linewidth=2)
axes[1, 1].plot(df['epoch'], df['val/dfl_loss'], label='DFL Loss', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Validation Losses')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: training_analysis.png")

print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"\nBest model saved at: {run_path}/weights/best.pt")
print(f"Latest model saved at: {run_path}/weights/last.pt")
print(f"\nConfusion matrix: {run_path}/confusion_matrix.png")
print(f"Results plot: {run_path}/results.png")
print(f"Training analysis: training_analysis.png")
print("="*60)

CHECKING TRAINING RESULTS
Location: ./runs/detect/train9/

TRAINING SUMMARY

Training Details:
  Total epochs completed: 68
  Final epoch: 68

Final Metrics:
  mAP@0.5: 0.6250 (62.51%)
  mAP@0.5:0.95: 0.4614 (46.14%)
  Precision: 0.5440
  Recall: 0.6775

Final Losses:
  Box loss: 0.7120
  Class loss: 0.6681
  DFL loss: 1.1801

Best Performance:
  Best mAP@0.5: 0.6422 at epoch 46
  Best mAP@0.5:0.95: 0.4723

GENERATING PLOTS
✓ Saved: training_analysis.png

RESULTS SUMMARY

Best model saved at: ./runs/detect/train9//weights/best.pt
Latest model saved at: ./runs/detect/train9//weights/last.pt

Confusion matrix: ./runs/detect/train9//confusion_matrix.png
Results plot: ./runs/detect/train9//results.png
Training analysis: training_analysis.png


In [27]:
from ultralytics import YOLO
import os
import numpy as np

print("="*60)
print("EVALUATING MODEL ON TEST SET")
print("="*60)

# ============================================ # FIND BEST MODEL # ============================================ 
model_path = "./runs/detect/train9/weights/best.pt" 

if not os.path.exists(model_path): 
    print(f"❌ Model not found at: {model_path}") 
    exit(1) 

print(f"Loading model: {model_path}\n")

# ============================================
# LOAD MODEL AND EVALUATE
# ============================================
model = YOLO(model_path)

print("Running evaluation on test set...")
print("This may take 2-5 minutes...\n")

results = model.val(
    data="data.yaml",
    split='test',           
    batch=16,
    imgsz=640,
    save_json=True,
    save_hybrid=True,
    conf=0.001,
    iou=0.6,
    plots=True,
    verbose=True
)

# ============================================
# PRINT RESULTS
# ============================================
print("\n" + "="*60)
print("TEST SET EVALUATION RESULTS")
print("="*60)

print(f"\nOverall Metrics:")
print(f"  mAP@0.5: {results.box.map50:.4f} ({results.box.map50*100:.2f}%)")
print(f"  mAP@0.5:0.95: {results.box.map:.4f} ({results.box.map*100:.2f}%)")
print(f"  Precision: {results.box.mp:.4f}")
print(f"  Recall: {results.box.mr:.4f}")

print(f"\nPer-Class Performance:")
print(f"  Classes evaluated: {len(results.box.ap50)}")
print(f"  Best class mAP@0.5: {results.box.ap50.max():.4f}")
print(f"  Worst class mAP@0.5: {results.box.ap50.min():.4f}")
print(f"  Mean class mAP@0.5: {results.box.ap50.mean():.4f}")
print(f"  Median class mAP@0.5: {np.median(results.box.ap50):.4f}")

print(f"\nInference Speed:")
print(f"  Preprocess: {results.speed['preprocess']:.2f} ms")
print(f"  Inference: {results.speed['inference']:.2f} ms")
print(f"  Postprocess: {results.speed['postprocess']:.2f} ms")
total_time = sum(results.speed.values())
print(f"  Total per image: {total_time:.2f} ms")
print(f"  FPS: {1000/total_time:.1f}")

print("\n" + "="*60)
print("PERFORMANCE ASSESSMENT")
print("="*60)

map50 = results.box.map50

if map50 >= 0.70:
    print("✓ EXCELLENT: mAP > 70% - Very strong performance!")
elif map50 >= 0.60:
    print("✓ GOOD: mAP 60-70% - Solid performance")
elif map50 >= 0.50:
    print("⚠ ACCEPTABLE: mAP 50-60% - Decent but could improve")
elif map50 >= 0.40:
    print("⚠ BELOW TARGET: mAP 40-50% - Consider retraining")
else:
    print("❌ POOR: mAP < 40% - Needs improvement")

print("\n" + "="*60)


EVALUATING MODEL ON TEST SET
Loading model: ./runs/detect/train9/weights/best.pt

Running evaluation on test set...
This may take 2-5 minutes...

WARNING 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.237  Python-3.10.9 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
YOLO11s summary (fused): 100 layers, 9,429,054 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 303.6177.7 MB/s, size: 38.1 KB)
val: Scanning C:\Users\nicho\Dokumen\Projects\FOOD\dataset_final\test\labels.cache... 1108 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1108/1108 2.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 70/70 6.8it/s 10.4s0.1s
                   all       1108       1795      0.566      0.678      0.645      0.475
               class_1         15         72      0.652      0.694       0.63      0.295
               class_2         17

In [29]:
from ultralytics import YOLO

# 1. Load your best trained weights
# Adjust path if yours is different (e.g., 'runs/detect/train/weights/best.pt')
model = YOLO('runs/detect/train9/weights/best.pt')

# 2. Run validation on the 'test' split
# The split='test' argument tells YOLO to look at the 'test:' path in data.yaml
metrics = model.val(
    data='data.yaml', 
    split='test',       # Key argument!
    imgsz=640,          # Ensure this matches your training image size
    batch=16,
    conf=0.25,          # Confidence threshold for evaluation
    iou=0.6,            # NMS IoU threshold
    plots=True          # Saves confusion matrices and plots
)

# 3. Print results
print(f"Mean Average Precision @.50:.95 : {metrics.box.map}")
print(f"Mean Average Precision @.50     : {metrics.box.map50}")

Ultralytics 8.3.237  Python-3.10.9 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
YOLO11s summary (fused): 100 layers, 9,429,054 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 699.7435.2 MB/s, size: 87.5 KB)
val: Scanning C:\Users\nicho\Dokumen\Projects\FOOD\dataset_final\test\labels.cache... 1108 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1108/1108  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 70/70 7.6it/s 9.2s0.1s
                   all       1108       1795      0.581       0.65      0.623       0.48
               class_1         15         72      0.658      0.667      0.654      0.349
               class_2         17         59      0.683      0.729      0.782      0.506
               class_3         46         48      0.763      0.938      0.886      0.655
               class_4         18         18      0.917      0.611      0.785    